# 第 8 週 實作｜瑕積分與收斂判定

定積分一直假設「有限區間 + 有界函數」。這週兩個假設都拿掉,然後你會看到:無限長的區域可以有有限面積,無限高的尖峰也可以。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜收斂與發散,用眼睛看

瑕積分的定義是「先算到 $b$,再讓 $b\to\infty$」。這格把 $\int_1^b$ 當成 $b$ 的函數畫出來,收斂與發散的差別一目了然。


In [ ]:
from scipy.integrate import quad

cases = [
    ("1/x^2   (p=2, 收斂)",  lambda t: 1/t**2,     1.0),
    ("1/x^1.5 (p=1.5, 收斂)", lambda t: t**-1.5,   2.0),
    ("1/x     (p=1, 發散)",   lambda t: 1/t,       None),
    ("1/sqrt(x)(p=0.5, 發散)", lambda t: t**-0.5,  None),
]

bs = np.logspace(0, 4, 60)          # b 從 1 掃到 10000
plt.figure(figsize=(7, 4.5))
for name, f, limit in cases:
    vals = [quad(f, 1, b)[0] for b in bs]
    plt.semilogx(bs, vals, label=name)
    if limit:
        plt.axhline(limit, ls=':', lw=1, color='gray')
plt.xlabel('b'); plt.ylabel(r'$\int_1^b f$')
plt.legend(fontsize=8); plt.title('Convergent tails flatten; divergent ones keep climbing')
plt.show()

print(f"{'被積式':22s} {'b=10':>10} {'b=100':>10} {'b=10^4':>10} {'理論值':>10}")
for name, f, limit in cases:
    row = [quad(f, 1, b)[0] for b in (10, 100, 1e4)]
    lim = f"{limit:10.4f}" if limit else "       發散"
    print(f"{name:22s} {row[0]:10.4f} {row[1]:10.4f} {row[2]:10.4f} {lim}")

In [ ]:
# TODO 學生練習:加一條 1/(x*log(x)) 從 b=2 開始積(觀念 3 的 D3)
# 它收斂還是發散?從圖上看得出來嗎?要掃到多大的 b 才看得出趨勢?

## Lab 2｜p 判別法:兩個方向相反的門檻

觀念 3、4 說兩個 $p$ 判別法的不等號方向<strong>相反</strong>。這格把兩邊的臨界行為一起算出來對照。


In [ ]:
from scipy.integrate import quad

print("=== 尾巴 [1, ∞):p > 1 才收斂,值 = 1/(p-1) ===")
print(f"{'p':>6} {'數值':>14} {'理論 1/(p-1)':>16} {'判定'}")
for p in [0.5, 1.0, 1.5, 2.0, 3.0]:
    if p > 1:
        v, _ = quad(lambda t, p=p: t**(-p), 1, np.inf)
        print(f"{p:6.1f} {v:14.6f} {1/(p-1):16.6f}   收斂")
    else:
        v, _ = quad(lambda t, p=p: t**(-p), 1, 1e6)
        print(f"{p:6.1f} {v:14.6f} {'—':>16}   發散(積到 1e6 仍在漲)")

print("\n=== 端點 (0, 1]:p < 1 才收斂,值 = 1/(1-p) ===")
print(f"{'p':>6} {'數值':>14} {'理論 1/(1-p)':>16} {'判定'}")
for p in [0.3, 0.5, 1.0, 1.5, 2.0]:
    if p < 1:
        v, _ = quad(lambda t, p=p: t**(-p), 0, 1)
        print(f"{p:6.1f} {v:14.6f} {1/(1-p):16.6f}   收斂")
    else:
        v, _ = quad(lambda t, p=p: t**(-p), 1e-6, 1)
        print(f"{p:6.1f} {v:14.6f} {'—':>16}   發散(從 1e-6 積起已很大)")

print("\n→ 同一個 p,在兩個區間上的命運相反。")
print("  p = 1 是唯一兩邊都發散的:1/x 兩頭不討好。")

## Lab 3｜歸一化:從高斯積分到 softmax

觀念 8、9 的數值驗證。先確認 $\int e^{-x^2}dx=\sqrt\pi$,再看連續版的 softmax(配分函數)什麼時候會爆掉。


In [ ]:
from scipy.integrate import quad

# --- 高斯積分 ---
v, err = quad(lambda t: math.exp(-t*t), -np.inf, np.inf)
print(f"∫ exp(-x^2) dx = {v:.12f}")
print(f"sqrt(pi)       = {math.sqrt(math.pi):.12f}   誤差 {abs(v-math.sqrt(math.pi)):.2e}")

# --- 標準常態的歸一化常數 ---
z, _ = quad(lambda t: math.exp(-t*t/2), -np.inf, np.inf)
print(f"\n∫ exp(-x^2/2) dx = {z:.12f}   sqrt(2pi) = {math.sqrt(2*math.pi):.12f}")
print(f"所以歸一化常數 c = 1/sqrt(2pi) = {1/math.sqrt(2*math.pi):.12f}")

# --- 配分函數:哪些能量合法 ---
print("\n配分函數 Z = ∫ exp(-E(x)) dx:")
energies = [
    ("E = x^2/2   (常態)",      lambda t: t*t/2,              True),
    ("E = |x|     (Laplace)",   lambda t: abs(t),             True),
    ("E = log(1+x^2) (Cauchy)", lambda t: math.log(1+t*t),    True),
    ("E = -x^2    (壞掉)",       lambda t: -t*t,               False),
]
for name, E, ok in energies:
    if ok:
        Z, _ = quad(lambda t, E=E: math.exp(-E(t)), -np.inf, np.inf)
        print(f"  {name:26s} Z = {Z:12.6f}   → 合法密度")
    else:
        Z, _ = quad(lambda t, E=E: math.exp(-E(t)), -5, 5)
        print(f"  {name:26s} Z(只積 [-5,5]) = {Z:.3e}   → 發散,不是密度")

# --- 離散 softmax 永遠合法 ---
zs = np.array([2.0, 1.0, 0.1, -3.0])
soft = np.exp(zs) / np.exp(zs).sum()
print(f"\n離散 softmax({zs}) = {np.round(soft, 6)}   總和 = {soft.sum():.10f}")

In [ ]:
# TODO 學生練習:試 E(x) = x^4/4。Z 收斂嗎?算出來是多少?
# 再試 E(x) = |x|^0.5,收斂嗎?(提示:先想尾巴衰減得夠不夠快)